# Checkpointing & Human-in-the-Loop (HITL)

LangGraph's **checkpointer** saves the full graph state after every node. This enables:
- **Multi-turn conversation** with persistent state across invocations
- **Time travel**: inspect or rewind to any past checkpoint
- **Human-in-the-loop**: pause execution, inject human input, resume

Topics:
- `MemorySaver` (in-memory, for development)
- `SqliteSaver` (on-disk, for production)
- `app.get_state()` and `app.get_state_history()`
- `interrupt_before=` — pausing a graph at a specific node
- `app.update_state()` — injecting human decisions
- Iterative review loop with HITL

In [ ]:
import operator, tempfile
from typing import Literal
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from typing_extensions import TypedDict, Annotated

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. MemorySaver — Multi-turn Conversation

Attach a `checkpointer` when compiling. Each `invoke` with the same `thread_id` continues the conversation — no need to pass previous messages manually.

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

def chat(state: ChatState) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

g = StateGraph(ChatState)
g.add_node("chat", chat)
g.add_edge(START, "chat")
g.add_edge("chat", END)

saver = MemorySaver()
app   = g.compile(checkpointer=saver)

# thread_id identifies the conversation session
config = {"configurable": {"thread_id": "user-123"}}

r1 = app.invoke({"messages": [HumanMessage(content="My name is Paulo")]}, config)
print(f"Turn 1: {r1['messages'][-1].content}")

# Second invoke — only sends the new message; history is restored from checkpoint
r2 = app.invoke({"messages": [HumanMessage(content="What's my name?")]}, config)
print(f"Turn 2: {r2['messages'][-1].content}")

state = app.get_state(config)
print(f"\nTotal messages in checkpoint: {len(state.values['messages'])}")

## 2. SqliteSaver — Persistence Across Restarts

Swap `MemorySaver` for `SqliteSaver` — same API, durable storage. Survives process restarts.

In [ ]:
with tempfile.NamedTemporaryFile(suffix=".db", delete=False) as f:
    db_path = f.name

print(f"SQLite database: {db_path}")

# Session 1
with SqliteSaver.from_conn_string(db_path) as saver:
    app2 = g.compile(checkpointer=saver)
    cfg  = {"configurable": {"thread_id": "persistent-user"}}
    app2.invoke({"messages": [HumanMessage(content="Remember: The secret code is ALPHA-7")]}, cfg)
    print("Session 1: Secret stored")

# Session 2 — fresh process, same DB
with SqliteSaver.from_conn_string(db_path) as saver:
    app3   = g.compile(checkpointer=saver)
    result = app3.invoke({"messages": [HumanMessage(content="What was the secret code?")]}, cfg)
    print(f"Session 2: {result['messages'][-1].content}")

## 3. State Inspection & Time Travel

Every node execution creates a checkpoint. You can inspect all of them and even **resume from a past checkpoint** (time travel).

In [ ]:
memory = MemorySaver()
app4   = g.compile(checkpointer=memory)
cfg4   = {"configurable": {"thread_id": "inspect-demo"}}

app4.invoke({"messages": [HumanMessage(content="Hello!")]}, cfg4)
app4.invoke({"messages": [HumanMessage(content="How are you?")]}, cfg4)

# Current state snapshot
state = app4.get_state(cfg4)
print(f"Next node:     {state.next or '() — finished'}")
print(f"Message count: {len(state.values['messages'])}")
print(f"Checkpoint ID: {state.config['configurable']['checkpoint_id'][:30]}...")

# Full history — one snapshot per node execution
print("\nCheckpoint history:")
for i, snap in enumerate(app4.get_state_history(cfg4)):
    writes = snap.metadata.get("writes") or {}
    node   = list(writes.keys())[0] if writes else "input"
    print(f"  [{i}] node={node:15s}  messages={len(snap.values['messages'])}  next={snap.next}")
    if i >= 4: print("  ..."); break

## 4. Human-in-the-Loop: Interrupt for Approval

**Pattern**: graph pauses before a designated node, human inspects state and injects feedback, then execution resumes.

```
START → [draft] → ⏸ PAUSE (interrupt_before=["approval"])
                        ↓  human calls app.update_state()
               → [approval] → [finalize] → END
```

Key APIs:
- `compile(checkpointer=..., interrupt_before=["node_name"])` — pause point
- `app.get_state(config)` — inspect frozen state
- `app.update_state(config, {...})` — inject human values
- `app.invoke(None, config)` — resume from checkpoint

In [ ]:
class ApprovalState(TypedDict):
    request: str
    draft: str
    approved: bool
    feedback: str
    final: str

def create_draft(state: ApprovalState) -> dict:
    resp = llm.invoke(f"Create a professional response for: {state['request']}")
    return {"draft": resp.content}

def wait_for_approval(state: ApprovalState) -> dict:
    return state  # just a placeholder — human injects approved/feedback via update_state

def finalize(state: ApprovalState) -> dict:
    if state["approved"]:
        return {"final": state["draft"]}
    resp = llm.invoke(f"Revise based on feedback:\n\nDraft: {state['draft']}\n\nFeedback: {state['feedback']}")
    return {"final": resp.content}

g = StateGraph(ApprovalState)
g.add_node("draft",    create_draft)
g.add_node("approval", wait_for_approval)
g.add_node("finalize", finalize)
g.add_edge(START, "draft")
g.add_edge("draft",    "approval")
g.add_edge("approval", "finalize")
g.add_edge("finalize", END)

memory = MemorySaver()
# interrupt_before pauses BEFORE this node runs
app5   = g.compile(checkpointer=memory, interrupt_before=["approval"])
config = {"configurable": {"thread_id": "hitl-demo"}}

# --- PHASE 1: Run until interrupt ---
result = app5.invoke({"request": "Write a thank-you email for a job interview", "draft": "", "approved": False, "feedback": "", "final": ""}, config)

print("Graph paused! Draft ready for review:")
print(result['draft'][:200], "...")

paused = app5.get_state(config)
print(f"\nNext node: {paused.next}  |  Final so far: '{result['final']}'")

In [ ]:
# --- PHASE 2: Human provides feedback ---
app5.update_state(config, {"approved": False, "feedback": "Make it more concise and mention company culture"})

# --- PHASE 3: Resume (None = no new input, just continue from checkpoint) ---
final_result = app5.invoke(None, config)

print("Final result after revision:")
print(final_result['final'][:300])

## 5. Iterative Review Loop (HITL + Cycles)

Human reviewers can approve or request changes multiple times. The graph loops back until the reviewer approves.

In [ ]:
class ReviewState(TypedDict):
    document: str
    review_comments: list[str]
    revision_count: int
    status: str

def submit_for_review(state: ReviewState) -> dict:
    return {"status": "pending_review"}

def apply_feedback(state: ReviewState) -> dict:
    if not state["review_comments"]:
        return state
    feedback = state["review_comments"][-1]
    resp = llm.invoke(f"Revise document based on feedback:\n\nDocument: {state['document']}\n\nFeedback: {feedback}")
    return {"document": resp.content, "revision_count": state["revision_count"] + 1, "status": "revised"}

def route_after_review(state: ReviewState) -> Literal["apply", "done"]:
    return "done" if state["status"] == "approved" else "apply"

def finalize(state: ReviewState) -> dict:
    return {"status": "finalized"}

g = StateGraph(ReviewState)
g.add_node("submit", submit_for_review)
g.add_node("apply",  apply_feedback)
g.add_node("done",   finalize)
g.add_edge(START, "submit")
g.add_conditional_edges("submit", route_after_review, {"apply": "apply", "done": "done"})
g.add_edge("apply", "submit")  # loop back
g.add_edge("done", END)

memory = MemorySaver()
app6   = g.compile(checkpointer=memory, interrupt_before=["submit"])
cfg6   = {"configurable": {"thread_id": "review-1"}}

# Initial submission
app6.invoke({"document": "AI is technology that helps computers think.", "review_comments": [], "revision_count": 0, "status": ""}, cfg6)
print(f"Paused. Next: {app6.get_state(cfg6).next}")

# Round 1: reviewer requests changes
app6.update_state(cfg6, {"review_comments": ["Add more technical depth and examples"], "status": "needs_revision"})
app6.invoke(None, cfg6)
print(f"Revised. Next: {app6.get_state(cfg6).next}")

# Round 2: reviewer approves
app6.update_state(cfg6, {"status": "approved"})
final = app6.invoke(None, cfg6)
print(f"Final status: {final['status']}  |  Revisions: {final['revision_count']}")
print(f"Document:\n{final['document'][:300]}")